In [0]:
import base64
import gzip
import hashlib
import io
import tarfile

dbutils.widgets.text("run_id", "")
dbutils.widgets.text("run_open_ts", "")
dbutils.widgets.text("silver_update_id", "")
RUN = dbutils.widgets.get("run_id").strip()
RUN_OPEN_TS = dbutils.widgets.get("run_open_ts").strip()
SILVER_UPDATE_ID = dbutils.widgets.get("silver_update_id").strip()
assert RUN.startswith("dq4_silver_") and RUN.replace("_", "").isalnum(), RUN
assert RUN_OPEN_TS and SILVER_UPDATE_ID
LANE = "dq4_issueify"
PAYLOAD = """H4sIAAAAAAAAA+1ZW2/jthLOs34FHw5AGbW1li/xtjhdIE3UbYqsndpOi+KkVWWJzqqRJUekspuDg/PbOyRl3eUkrbOLovweEkkcDofDuXHsU5oQf/1gU7Zh1L43Xx0dHH3AZDLm/83JuF/8v8OROR6MhsP+yBxOjvrmqH88OkLjw4tSR0KZEyN0RBwWhat2usfG/6bwa+fv3Y3s7Cs/G4PeBX9pDc7kGA605fzHI9OsnP/keDw+Qv0D7XEv/uHn3+uhsx9GiLI4cVkSO8ErN44o7dEoiV2ChCH0/LXvOsyPQkMD+jkJPRIjbifUD+5JbA/6g+P+68G4i/hTr/+6x5+d0EPrsTtxJ86qR4YTrzdyhmZv9aU76K0mx0NzRdyVeeygFVlHMUHkI3ETuYr2zpq/tdD5dDlDv722PXL/m5Gudeca3p00UMS0q8X59C3SNYQW1oV1ukT0vTPQ3SgEee0PVMf/w10UG+574t7avtdFbuQEhLpEvzXg3G8Is5mzCkgX4WkUEtzpAqtm1Ge6UZBswnwqwv/nfwfj4w46WUjd8UVzlkVJqgJU2eazTk8WS316dXHBuS6Wc9iyWCBwQmLfJU4A50PiwgS5fxB07Wz84AEE+wpV9IDR2vED4iEd17csFoyNDXFoEhPPjqMPtLA2nx2tUftEFjEnaJrFP3WwEJ75DHadcwidDawk7VDHpaWF6KUvbaeE85XFpPy1dYaz3cbRR3/jMIIL5hEbhYEuKCugpNU4cnEdBkyEFnDuCpjv9+xkabUzkO5mJ1sPloMDAi74Kb4jdbmNyT3IHbqkbDUW+uk7awqWRQm4js8ewJPWThIw9DXCl32MlnyYP1XkEtOqBvDma2TKiF3bxWxeo36FwiQAyyyZQxf1O5xP3+ibTUxaJDUzSc1nSXoIOfvPknSQSTqoSmpdwHngyyFG1vSMn1rGgIH3AvG389m75mgn3TYmlC/DSb+fnU/3kKJbNOPHvnN3EC33fU0obW7BpzgJ5ShuCOZgtSBnbEB+ZAnlRDxiyF3JETeGpAh6XEH1QCiFyM2pIJ2AVVINJGDGLgbCAM1eNHFo706Wp99ZZ1JfV5fcPyCIL4E/MwKHMuBJQhtEbJOvC4S55YsVyo7AjJKKBUXpS0q02yD3GC0zqsIAdoOIEm93ttGWhLiZMIyY7YQyPUkF6qkZZJGlKHQlqqGe2UH/fpPT0kdoZ/O93IsGvZd1lbCJb1lzCOMqvwaCJj4y8DfMzwc6NUUL58kUDQ4kj07GSx52QfluEsckhHzqbwgQbrZ6R1radLYsW9v5dGHNuaXpWY5GeWosp+VKrq/k3EIaW/txwWg5Zem1YJqooim5K8HEI16yJbYbQFkKts7FIfe+x6fZ7x36votuTOAXSJnh2XF5xSTm1t2xm31zWBflyupoP55cXFkLpOdOCWIUygNaKU5oW3FCjapG8oNsdtqWz1Xn3bGvWpWwiS7i5VDTXxF+8vkNNtFt+tjRPnch/pnw+P1v8NL3v/5oMq7d//j9X93/Xh7Pu2jVSGy22RrCoV1wTLvBtXeVQKEWyQNNuRLg6bKa+yFFNAYMDaXkaUqAekjfxQbMYt+5IaKI5sED4pJ4jqME4h/uaJWSoynVd5+SXT734R0Aj/v/8KX9fzgxJ3X/nyj//xQ4nVvcC+BGM7cuL05OLbQ8+ebC2uPpqyT0gmZXP1loaRvGN2rdDxbZv9Mo1EsX/fyOhHcTwPUKs8GZeTUhPu7qimJpIgeKtUqBZalmKVJmnZty8SJIWrsquFSKCNpqcZIXZ2K4UKsV2GRV244mreEQzisgMZS/dgoNJa7E/UrFu/CKu4VLIOfvxM6Giq/yUfKVFI18Y8MjDK59gk4+PkGAwma3ceTBJ96NITfk49YmH1kMG9b9av/t15HNia+vU5uDB/0/19f0+nrxyxedfwEHs9g/was4Cv9LbBHU+Z5oFMPdK46dBx2ONyAuzyIs7QXqgbEmIKBsffXe8HZYYKRdFyFBh0N01fyQQPYob7ONuQHsGWwJGKe6zN5pTUctTJpaNkXG6HwhrjC8wK30FEQiq5x9No+3swpseEfJESfhGvyhw69RnZLUVMiptTciZDXga8/vP/h5zn9kcrHPMZsWGhclLmkLYn8HQ3u0UaFdWN8u29opqe3aqUWgQEoUZAO35IG3Vg5g2HvlAM3chtGH0E6PCblSEJf3KAOfUJtFogDaWcdBXS11oAO7r/Rho9hyrUdn3KmJgWVL+ysRxORzR1zz55aI3U/oGwmjuDVIeAPnyomgvsHa2/ns6hJ983Mp+WQ5p7rdqqz5xmoJpJ4mSomhlAKKbIoNg9vS7wYyducK4J1uGZufX5E+Xv+ZL37/Gw6G9d//hqr++xR4fv3XftPLy7+z88XyfAoPWbh+WnM7deQn9KX3hPTPrdK/FR73/9HL+/9oVL//mcr/PwUO1f8hvDu9r//zJ38JKnW/xdxKP/yf0aZRUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFAQ+ANSB+ctAFAAAA=="""

def qs(value):
    if value is None:
        return "NULL"
    return "'" + str(value).replace("'", "''") + "'"

def adapt(value):
    return (value
        .replace("dq4_silver_20260825", RUN)
        .replace("2026-08-25T10:50:49.897Z", RUN_OPEN_TS)
        .replace("f5c7c7ab-e37d-4a31-b9c2-b7631becb16a", SILVER_UPDATE_ID))

def unpack():
    archive = tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode="r:gz")
    items = []
    for member in archive.getmembers():
        if member.isfile() and member.name.endswith(".sql"):
            items.append((member.name, adapt(archive.extractfile(member).read().decode("utf-8"))))
    return sorted(items, key=lambda x: x[0])

def execute(seq, name, sql):
    sha = hashlib.sha256(sql.encode()).hexdigest()
    spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log
      (run_id,lane,stmt_file,seq,stmt_sha256,status,error_text,rows_affected,attempted_at,settled_at,created_by_session)
      VALUES ({qs(RUN)},{qs(LANE)},{qs(name)},{seq},{qs(sha)},'attempted',NULL,NULL,current_timestamp(),NULL,'DQ4')""")
    try:
        spark.sql(sql).collect()
        spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log
          (run_id,lane,stmt_file,seq,stmt_sha256,status,error_text,rows_affected,attempted_at,settled_at,created_by_session)
          VALUES ({qs(RUN)},{qs(LANE)},{qs(name)},{seq},{qs(sha)},'ok',NULL,NULL,current_timestamp(),current_timestamp(),'DQ4')""")
    except Exception as exc:
        msg = str(exc)[:4000]
        spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log
          (run_id,lane,stmt_file,seq,stmt_sha256,status,error_text,rows_affected,attempted_at,settled_at,created_by_session)
          VALUES ({qs(RUN)},{qs(LANE)},{qs(name)},{seq},{qs(sha)},'error',{qs(msg)},NULL,current_timestamp(),current_timestamp(),'DQ4')""")
        raise

run_row = spark.sql(f"SELECT count(*) n FROM 8_dev.silver_qc.dq_run WHERE run_id={qs(RUN)} AND finished_at IS NULL").first().n
assert run_row == 1, "run_id must identify one open dq_run row"

items = unpack()
for seq, (name, sql) in enumerate(items):
    execute(seq, name, sql)

print({"run_id": RUN, "lane": LANE, "statements": len(items), "status": "ok"})